In [ ]:
import scripts.helpers as helpers

1. Step 1 - Create unified labels file -> in parquete format
2. Step 2 - Unify each SWC tree with it's labels
3. Step 3 - Simplify SWC tree
4. Step 4 - Calculate clumpiness for each node in the tree, for each labels combination
5. Step 5 - Insert the clumpiness score  into the simplified SWC file

---
# Step 1 - Create unified labels file -> in parquete format

---
# Step 2 - Unify each SWC tree with it's labels

---
# Step 3 - Simplify SWC tree

---
# Step 4 - Calculate clumpiness for each node in the tree, for each labels combination

---
# Step 5 - Insert the clumpiness score  into the simplified SWC file

---
# Preprocessing step
1. Create metadata labels for each swc file
2. Look for the releveant swc files only (with the wanted type)
3. Unify them via the already created function in feather file ->>> Improvement

In [43]:
import os
import json
import pandas as pd
import polars as pl
from tqdm import tqdm
from scripts.helpers import mkdir
from scripts.preprocessing import simplify_swc_topology


# Type data located in the 

path_swc_labels = os.path.join("data", "input_labels", "neuron_data_full_article_princeton.ftr")
swc_labels = pd.read_feather(path_swc_labels)

required_labels = ["super_class", ["central", "optic", "visual_centrifugal", "visual_projection"]]
swc_labels = swc_labels.loc[swc_labels[required_labels[0]].isin(required_labels[1])]

In [44]:
#  0. Get the list of required swc files
#  1. Load labels parquet
#  2. Import the swc file
#  3. simplify swc file
#  4. attach synapse labels + neuron type
#  5. save simplified file
#  6. convert to json for find-clumpiness
#  7. save json file
#  8. calculate clumpiness for each internal node
#  9. attach results to the labeled swc file
# 10. save results.

In [49]:
###########################
###########################
def wsc2json(swc_dataset : str,
             neuron_id: str ,
             save_json : bool = False,
             save_path: str = None,
             print_msg : bool = False) -> None:
    """
    Converts SWC and FTR files into a JSON structure suitable for find-clumpiness.
    swc_dataset : str -> SWC information with labels attached (synpase annotations).
    neuron_id: str -> Neuron name (id).
    save_json : bool -> If True will save the JSON file into the path stated in the `output_folder`.
    output_folder: str -> Folder to which save the processed json file.
    print_msg : bool = False -> if True will print messege about the processed neuron.
    """
    
    # Load csv or import pd.DataFrame object
    if isinstance(swc_dataset, pd.DataFrame):
        swc_df = swc_dataset

    elif isinstance(swc_dataset, str):
        try:
            swc_df = pd.read_csv(swc_dataset, index_col=0)
        except:
            raise Exception("> Invalid string input for `swc_dataset` argument")
        

    swc_df["node_id"] = swc_df["node_id"].astype(int)
    swc_df["parent"] = swc_df["parent"].astype(int)


    # Preparing for the json tree construction
    children_map = {}
    node_labels = {}
    root = None
        
    for _, row in swc_df.iterrows():
        node = str(int(row['node_id']))
        parent_val = row['parent']
            
         # Labels are already aggregated into a list from the groupby
        # If label found -> add to the labels dicts with the node_id as key
        labels = row['type']
        if isinstance(labels, list):
            node_labels[node] = labels
        else:
             node_labels[node] = []
                
         # Handle topology and find the root
        # Assigning root nodes
        if pd.isna(parent_val) or parent_val == -1: 
            root = node
            
        # Assigning rest of nodes
        else:
            parent = str(int(parent_val))
            if parent not in children_map:
                children_map[parent] = []
            children_map[parent].append(node)

    if print_msg:
        print("Neuron tree mapped.")
        
    # Incase there is no defined root note with parent values of -1.
    if root is None:
        raise ValueError("Could not find the root node (a node where parent is -1).")
            
    # Anti-infinite loop section, preventing from `node -> parent`, `parent -> node` loop to occure
    # List of visited nodes
    itirated = set()

    def build_node(node_id):
        # Stop execution if returning to previously visited node
        if node_id in itirated:
            raise RecursionError(f"Cycle detected in SWC file at node {node_id}. Fix the source data.")
        itirated.add(node_id)
            
        node_dict = {"nodeID": node_id,
                     "nodeLabels": node_labels.get(node_id, [])} # Returning the node label, if not found returns empty list
            
        children_list = []
        if node_id in children_map:
            for child_id in children_map[node_id]:
                children_list.append(build_node(child_id))
                    
        return [node_dict, children_list]
        
    # Build JSON, from the parent node to the leaves.
    final_json = build_node(root)
        
    # Export
    if save_json & isinstance(save_path, str):
        path_real = os.path.exists(save_path)
        file_path = os.path.join(save_path, f"{neuron_id}.json")

        if os.path.exists(save_path) is False:
            os.mkdir(save_path)

        with open(file_path, 'w') as f:
            json.dump(file_path, f, separators=(',', ':'))


In [56]:
import os
import json
import pandas as pd
import ast

def wsc2json(swc_dataset,
             neuron_id: str,
             save_json: bool = False,
             save_path: str = None,
             print_msg: bool = False) -> None:
    """
    Converts SWC and FTR files into a JSON structure suitable for find-clumpiness.
    swc_dataset : str or pd.DataFrame -> SWC information with labels attached.
    neuron_id: str -> Neuron name (id).
    save_json : bool -> If True will save the JSON file into the path stated in `save_path`.
    save_path: str -> Folder to which to save the processed json file.
    print_msg : bool = False -> if True will print message about the processed neuron.
    """
    
    # Load csv or import pd.DataFrame object
    if isinstance(swc_dataset, pd.DataFrame):
        swc_df = swc_dataset.copy()
    elif isinstance(swc_dataset, str):
        try:
            swc_df = pd.read_csv(swc_dataset, index_col=0)
        except Exception as e:
            raise Exception(f"> Failed to load CSV: {e}")
    else:
        raise ValueError("> Invalid input type for `swc_dataset` argument")

    # Fix 1: Auto-drop duplicates to prevent RecursionError cycles
    if swc_df.duplicated(subset=['node_id']).any():
        swc_df = swc_df.drop_duplicates(subset=['node_id'], keep='first')

    # Fix 2: Safely convert to int by filling NaNs with -1 first
    swc_df["node_id"] = swc_df["node_id"].fillna(-1).astype(int)
    swc_df["parent"] = swc_df["parent"].fillna(-1).astype(int)

    # Preparing for the json tree construction
    children_map = {}
    node_labels = {}
    root = None
        
    for _, row in swc_df.iterrows():
        node = str(int(row['node_id']))
        parent_val = row['parent']
            
        labels = row['type']
        
        # Fix 3: Robust label parsing
        if pd.isna(labels):
            node_labels[node] = []
        elif isinstance(labels, str):
            # Check if it's a string representation of a list
            if labels.startswith('[') and labels.endswith(']'):
                try:
                    node_labels[node] = ast.literal_eval(labels)
                except (ValueError, SyntaxError):
                    node_labels[node] = [labels]
            else:
                node_labels[node] = [labels]
        elif isinstance(labels, list):
            node_labels[node] = labels
        else:
            node_labels[node] = []
                
        # Handle topology and find the root
        if pd.isna(parent_val) or parent_val == -1: 
            root = node
        else:
            parent = str(int(parent_val))
            if parent not in children_map:
                children_map[parent] = []
            children_map[parent].append(node)

    if print_msg:
        print(f"Neuron tree mapped for {neuron_id}.")
        
    # In case there is no defined root node with parent values of -1
    if root is None:
        raise ValueError("Could not find the root node (a node where parent is -1).")
            
    # Anti-infinite loop section
    itirated = set()

    def build_node(node_id):
        if node_id in itirated:
            raise RecursionError(f"Cycle detected in SWC file at node {node_id}. Fix the source data.")
        itirated.add(node_id)
            
        node_dict = {
            "nodeID": node_id,
            "nodeLabels": node_labels.get(node_id, [])
        }
            
        children_list = []
        if node_id in children_map:
            for child_id in children_map[node_id]:
                children_list.append(build_node(child_id))
                    
        return [node_dict, children_list]
        
    # Build JSON, from the parent node to the leaves
    final_json = build_node(root)
        
    # Export
    # Fix 4: Changed bitwise '&' to logical 'and'
    if save_json and isinstance(save_path, str):
        # Fix 5: Ensure the directory exists correctly without throwing an error if it does
        os.makedirs(save_path, exist_ok=True)
        
        file_path = os.path.join(save_path, f"{neuron_id}.json")

        with open(file_path, 'w') as f:
            # Fix 6: Dumping the final_json data, NOT the file_path string
            json.dump(final_json, f, separators=(',', ':'))

In [65]:
dataset = simple_swc
dataset[dataset.node_id == 1722]

,node_id,swc_type,x,y,z,r,parent
132,1722,6,333527.53,227110.97,265214.1,0,1846


In [66]:
nueron_itr = 720575940609102805


####################################################################################################
#  1. Load labels parquet
# Parquet labels path
prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

# Load exactly the labels of the example swc file
parquet_labels = pl.scan_parquet(prquet_labels_path)

# Only the relevnt column in the parquet file
labels_parquet = parquet_labels.select(["neuron", "node_id", "type"]).filter(pl.col("neuron") == str(nueron_itr)).collect().to_pandas()


####################################################################################################
#  2. Import the swc file
neuron_path = os.path.join("data","input_swc", "sk_lod1_783_healed", f"{nueron_itr}.swc")
neuron_swc = pd.read_csv(neuron_path, 
                         comment='#', 
                         header=None, 
                         sep=r'\s+', 
                         names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])


####################################################################################################
#  3. simplify swc file
simple_swc = simplify_swc_topology(neuron_swc, swc_name=f"{nueron_itr}", save_csv=False)


####################################################################################################
#  4. attach synapse labels + neuron type
swc_labeled = pd.merge(left=simple_swc, 
                       right=labels_parquet[["node_id", "type"]], 
                       left_on="node_id", 
                       right_on="node_id", 
                       how="inner")


####################################################################################################
#  5. save simplified file
for i in ["data", os.path.join("data", "input_swc"), os.path.join("data", "input_swc", "simplified")]:
    if os.path.exists(i) is False:
        os.mkdir(i)

save_path = os.path.join("data", "input_swc", "simplified", f"{nueron_itr}.csv")
swc_labeled.to_csv(save_path)


####################################################################################################
#  6. convert to json for find-clumpiness
wsc2json(swc_dataset=swc_labeled,
         neuron_id=nueron_itr,
         save_json=True,
         save_path=os.path.join("data", "output_json"))


ValueError: Could not find the root node (a node where parent is -1).